# Step-projected three-step GC area comparison

This experiment evaluates the BM4-based method that projects two uncoupled guiding-center copies onto their mean after every direct or adjoint map. A complete BM4 step therefore contains twelve projections and uses no harmonic numerical coupling.

The study uses one 16-point circular boundary and three step sizes: $\pi/40$, $\pi/80$, and $\pi/160$. States and projected diagnostics are synchronized every $\pi/8$ up to $4\pi$.

The final animation combines the transported contours, relative area error, projected symplectic defect, and relative separation of the two internal GC copies.

In [1]:
import numpy as np

from workflows import (
    AreaComparisonConfig,
    RandomPotentialConfig,
    centered_circle,
    display_animation,
    pi_area_steps,
    run_area_comparison,
)

In [2]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()
circle_radius = 0.5
circle_points = 16
rho = 0.3
circle = centered_circle(
    potential,
    radius=circle_radius,
    points=circle_points,
    rho=rho,
)
comparison_config = AreaComparisonConfig(
    steps=pi_area_steps(40, 80, 160),
    t_span=(0.0, 4 * np.pi),
    save_interval=np.pi / 8,
    coupling_frequency=0.0,
    method_kind="stage_projected_bm4",
    progress=True,
)

print(potential_config)
print(
    f"Circle: {circle_points} points; t={comparison_config.t_span}; "
    f"{comparison_config.output_sample_count} saved states"
)

RandomPotentialConfig(amplitude=0.7, max_wave_number=25, nx=64, ny=64, seed=27, interpolation_order=5)
Circle: 16 points; t=(0.0, 12.566370614359172); 33 saved states


## Integrations and projected observations

The workflow derives the output grid, diagnostic stride, observer lifecycle, metadata blocks, and result mappings from the configuration above.

In [3]:
result = run_area_comparison(
    potential,
    circle,
    notebook_path=(
        "notebooks/experiments/symplecticity/"
        "gc_area_and_projected_symplecticity_3.ipynb"
    ),
    config=comparison_config,
    metadata={
        **potential_config.metadata(),
        "circle_radius": circle_radius,
    },
)
result.print_summary()

ProjectedBM4Composition [==============================] 100.0% (160/160, t=12.5664)
ProjectedBM4Composition [==============================] 100.0% (320/320, t=12.5664)
ProjectedBM4Composition [==============================] 100.0% (640/640, t=12.5664)


                  step integration steps     max |area error|      max symplectic defect    max relative separation
     $\Delta t=\pi/40$          160       1.77845982e-02             5.12486742e-06             0.00000000e+00
     $\Delta t=\pi/80$          320       1.77843997e-02             6.42093822e-07             0.00000000e+00
    $\Delta t=\pi/160$          640       1.77843746e-02             8.09948206e-08             0.00000000e+00


## Comparative animation

The same color identifies each integration step in all four synchronized panels.

In [4]:
display_animation(
    result.animate(
        frames=None,
        interval=120,
    )
)

## Interpretation

All three runs use exactly the same potential, 16-vertex circle, initial condition, and observation times. The internal copies are re-embedded on the diagonal after every direct or adjoint map, so their separation after every internal stage should be zero up to round-off. The projection is non-invertible, therefore the projected symplectic defect is a diagnostic to evaluate rather than a quantity this method is expected to preserve. Differences between curves isolate the step-size behavior of this stage-projected formulation.